In [76]:
import os
import sys
import random
import json
import pandas as pd
import pandas as pd
import numpy as np
import textwrap
from datetime import datetime, timedelta

In [77]:
def choose_email_type():
    # Define the probabilities for each type of email
    probabilities = {
        'internal_cardiff': 0.6,  # 60% chance for internal emails
        'from_cardiff': 0.25,     # 25% chance for emails from Cardiff
        'to_cardiff': 0.15        # 15% chance for emails to Cardiff
    }
    
    # Generate a random number between 0 and 1
    random_number = random.random()
    
    # Determine the email type based on the random number and the defined probabilities
    cumulative_probability = 0.0
    for email_type, probability in probabilities.items():
        cumulative_probability += probability
        if random_number < cumulative_probability:
            return email_type

# Example usage
for _ in range(10):  # Generating 10 random selections to illustrate
    print(choose_email_type())

internal_cardiff
internal_cardiff
internal_cardiff
from_cardiff
internal_cardiff
internal_cardiff
from_cardiff
internal_cardiff
from_cardiff
internal_cardiff


In [78]:
# Function to load JSON data from a file
def load_json(filename):
    with open(filename, 'r') as file:
        return json.load(file)


def generate_email_dates(year=1983, max_weekend_emails=5, max_weekday_emails=70):
    
    # Define the year of interest
    year = 1983
    
    # Generate all dates for the year 1983
    start_date = datetime(year, 1, 1)
    end_date = datetime(year, 12, 31)
    dates = pd.date_range(start=start_date, end=end_date, freq='D')
    
    # Define major US holidays in 1983 (New Year's Day, Independence Day, Thanksgiving, Christmas)
    holidays = [
        datetime(year, 1, 1),
        datetime(year, 7, 4),
        datetime(year, 11, 24),  # Typically Thanksgiving
        datetime(year, 12, 25)
    ]
    
    # Filter out the holidays
    dates = dates[~dates.isin(holidays)]
    
    # Email generation simulation
    email_counts = []
    for date in dates:
        # Check if it's a weekend
        if date.weekday() >= 5:  # 5 for Saturday, 6 for Sunday
            emails_today = np.random.randint(1, max_weekend_emails)
        else:
            emails_today = np.random.randint(1, max_weekday_emails)
        
        for _ in range(emails_today):
            # Generate a random time during typical business hours (9 AM to 5 PM)
            random_hour = np.random.randint(9, 17)
            random_minute = np.random.randint(0, 60)
            random_time = date.replace(hour=random_hour, minute=random_minute)
            
            # Store the date and email count
            email_counts.append(random_time.strftime("%A, %B %d, %Y %I:%M %p"))

    return email_counts

    

def choose_number_of_recipients():
    # Define the probabilities for the number of email recipients
    probabilities = {
        1: 0.35,   # 35% chance for 1 email recipient
        2: 0.15,   # 15% chance for 2 email recipients
        3: 0.15,   # 15% chance for 3 email recipients
        4: 0.15,   # 15% chance for 4 email recipients
        5: 0.15,
        6: 0.01,
        7: 0.01,
        8: 0.01,
        9: 0.01,
        10: 0.01
    }
    
    # Generate a random number between 0 and 1
    random_number = random.random()
    
    # Determine the email type based on the random number and the defined probabilities
    cumulative_probability = 0.0
    for number, probability in probabilities.items():
        cumulative_probability += probability
        if random_number < cumulative_probability:
            return number

# Function to randomly pick a character and associated attributes
def pick_email_sender_details():

    # Randomly pick a character
    characters = load_json('main_cardiff_characters.json')
    character = random.choice(characters)

    # Randomly pick a topic the character writes about
    topic = random.choice(character['Writes About'])
    
    # Randomly pick a writing emotion
    emotions = load_json('writing_emotions.json')
    emotion = random.choice(emotions['emotions'])

    return {
        'Name': character['name'],
        'Email': character['Email'],
        'Title': character['Title'],
        'Company': character['Company'],
        'Department': character['Department'],
        'Phone': character['Phone'],
        'Quote': character['Quote'],
        'Writes About': topic,
        'Personality Trait': character['Personality Trait'],
        'Writing Style': character['Writing Style'],
        'Emotion': emotion
    }

def pick_email_receivers(email_sender):
    
    # Chose number of email recipients
    number_recipients = choose_number_of_recipients()
    
    # Initialize list with email_sender so that person is not picked. 
    do_not_pick = [email_sender['Name']] 
    email_receivers = []

    # Pick random email recipients 
    for i in range(0,number_recipients):
        unpicked_characters = [person['name'] for person in load_json('main_cardiff_characters.json') if person['name'] not in do_not_pick]
        next_pick = random.choice(unpicked_characters)
        do_not_pick.append(next_pick)
        email_receivers.append(next_pick)
        
    return email_receivers
        
def make_receiver_string(email_receivers):
    receiver_str = ""
    for receiver in email_receivers:
        receiver_str = f"{receiver_str}" + receiver + "; "
    receiver_str = receiver_str.rstrip("; ")
    return receiver_str

def create_first_email_text_prompt(email_sender_dict, email_to_str):

    text_prompt =  (
        "This time is 1983, and the setting is the first season of the tv series Halt and Catch Fire. \n"
        "Let's pretend they can write email.\n\n" 
        "Let's please make the emails short and professional and write using the voice style reflected in the personality of the character in the tv show.\n"
        "{} is {} from {} in the {} department and wants to write an email about this category: {}.\n"
        "Their writing style can be described as {}.\n"
        "However, this person has a {} personality, and they are feeling {} as they write the email.\n\n"
        "They are writing this email to the following individual(s): \n{}\n\n"
        "Please write the body of the email in the voice of the character as indicated in this description.\n"
        "Please write the email in Outloook.txt format.\n"
        "Please DO NOT include Attachments:, Categories:. To:, From:, BCc, or Cc. I will add that afterwards.\n"
        "Please start the email with 'EMAIL START' \n"
        "Please end the email with 'EMAIL END'  \n"
        "The email should have four parts:\n"
        "The first part is the subject line which will follow 'Subject:    '. Please put a new line between the Subject: and the greeting.\n"
        "The second part is the greeting. You can use 'Dear','Hi','Hello', or whatever you deem is appropriate for the kind of conersation you are writing. \n"
        "The third part is the body of the email.\n"
        "The fourth part is the signature block.\n"
        "The signature block needs to be composed of the following elements:\n" 
        "The sender's name as {}\n"
        "The phone number is {}.\n"
        "Title is {}.\n"
        "Company is {}.\n"
        "The department is {}.\n"
        "As a reminder, please start the email with 'EMAIL START' \n"
        "As a reminder, please end the email with 'EMAIL END'  \n" 
        "Thank you so much for your help on this! I really appreciate it!").format(
        email_sender_dict['Name'], 
        email_sender_dict['Title'], 
        email_sender_dict['Company'],
        email_sender_dict['Department'],
        email_sender_dict['Writes About'],
        email_sender_dict['Writing Style'].lower(),
        email_sender_dict['Personality Trait'].lower(),
        email_sender_dict['Emotion'].lower(),
        email_to_str,
        email_sender_dict['Name'],
        email_sender_dict['Phone'],
        email_sender_dict['Title'],
        email_sender_dict['Company'],
        email_sender_dict['Department']
    )
    
    return text_prompt

def generate_email_body_llm_response(text_prompt):
    from langchain.llms import Ollama
    ollama = Ollama(base_url="http://localhost:11434", model="llama3")
    email_body = ollama(text_prompt)
    return email_body

def extract_email_body(email_text):

    # Define the markers for the start and end of the email
    start_marker = "EMAIL START"
    end_marker = "MAIL END"
    
    # Find the index of the start and end markers
    start_index = email_text.find(start_marker) + len(start_marker)
    end_index = email_text.find(end_marker)
    
    # Extract the email body using the indices found
    email_body = email_text[start_index:end_index].strip()

    return email_body


def build_email_header(email_sender, sent, email_to_str):
    header_text = (
        "\n"
        "From:    {}\n"
        "Sent:    {}\n"
        "To:      {}\n\n"
        "Categories: {}\n\n"
    ).format(email_sender['Name'], sent, email_to_str, email_sender['Writes About'])
    return header_text

def combine_header_body(header_text, email_body):
    email_text = "{}{}".format(header_text, email_body) 
    return email_text

def ensure_blank_lines_around_categories(email_text):
    # Split the email text into lines
    lines = email_text.split('\n')
    
    # Find the index of the line containing "Categories:"
    for i, line in enumerate(lines):
        if "Categories:" in line:
            # Ensure there's a blank line before "Categories:" if not already present
            if i > 0 and lines[i - 1].strip() != "":
                lines.insert(i, "")
            # Ensure there's a blank line after "Categories:" if not already present
            if i + 1 < len(lines) and lines[i + 1].strip() != "":
                lines.insert(i + 2, "")
            break
    
    # Join the lines back into a single string
    modified_email = '\n'.join(lines)
    return modified_email


def wrap_email_text(email_text, max_width=72):
    """
    Wraps the body of an email while preserving the formatting of headers and structured data.
    
    Args:
    email_text (str): The complete email text.
    max_width (int): The maximum number of characters in each line of the email body.
    
    Returns:
    str: The email with the body text wrapped according to the max_width.
    """
    lines = email_text.split('\n')
    wrapped_lines = []
    in_body = False  # To check if we are in the email body

    for line in lines:
        # Check if the line is part of the email headers or other structured parts
        if line.startswith("From:") or line.startswith("Sent:") or line.startswith("To:") or line.startswith("Cc:") or line.startswith("Bcc:") or line.startswith("Subject:") or line.startswith("Categories:") or line.strip().startswith("["):
            wrapped_lines.append(line)
            in_body = False
        else:
            if not in_body:
                # The first line of the email body after the headers
                in_body = True
                wrapped_lines.append("")
            
            # Wrap the body lines using textwrap
            wrapped_text = textwrap.fill(line, width=max_width)
            wrapped_lines.append(wrapped_text)

    return '\n'.join(wrapped_lines)

def save_text_to_file(text, filename="example.txt"):
    """
    Save the given text to a text file.
    
    Args:
    text (str): Text to be saved to the file.
    filename (str): The name of the file to save the text in.
    """
    with open(filename, "w", encoding="utf-8") as file:
        file.write(text)

def read_text_from_file(filename="example.txt"):
    """
    Read and return the text from a text file.
    
    Args:
    filename (str): The name of the file to read the text from.
    
    Returns:
    str: The text read from the file.
    """
    try:
        with open(filename, "r", encoding="utf-8") as file:
            return file.read()
    except FileNotFoundError:
        return "The file does not exist."
        
def generate_initial_email(sent):
    email_sender = pick_email_sender_details()
    email_receivers_list = pick_email_receivers(email_sender)
    email_to_str = make_receiver_string(email_receivers_list)
    text_prompt = create_first_email_text_prompt(email_sender, email_receivers_list)
    email_gen = generate_email_body_llm_response(text_prompt)
    email_body = extract_email_body(email_gen)
    header_text = build_email_header(email_sender, sent, email_to_str)
    email_text = combine_header_body(header_text, email_body)
    final_email = ensure_blank_lines_around_categories(email_text)
    final_email = final_email.rstrip('E')
    return final_email, email_sender

# Function to randomly pick a character and associated attributes
def pick_email_sender_details():

    # Randomly pick a character
    characters = load_json('main_cardiff_characters.json')
    character = random.choice(characters)

    # Randomly pick a topic the character writes about
    topic = random.choice(character['Writes About'])
    
    # Randomly pick a writing emotion
    emotions = load_json('writing_emotions.json')
    emotion = random.choice(emotions['emotions'])

    email_sender_dict = {
        'Name': character['name'],
        'Email': character['Email'],
        'Title': character['Title'],
        'Company': character['Company'],
        'Department': character['Department'],
        'Phone': character['Phone'],
        'Quote': character['Quote'],
        'Writes About': topic,
        'Personality Trait': character['Personality Trait'],
        'Writing Style': character['Writing Style'],
        'Emotion': emotion
    }

    return email_sender_dict 

In [79]:
def generate_email_chain(sent, num_replies):
    
    email_sender = pick_email_sender_details()
    email_receivers_list = pick_email_receivers(email_sender)
    email_to_str = make_receiver_string(email_receivers_list)
    
    text_prompt = create_first_email_text_prompt(email_sender, email_receivers_list)
    email_gen = generate_email_body_llm_response(text_prompt)
    email_body = extract_email_body(email_gen)
    header_text = build_email_header(email_sender, sent, email_to_str)
    email_text = combine_header_body(header_text, email_body)
    final_email = ensure_blank_lines_around_categories(email_text)


    return final_email, email_sender

In [80]:
def determine_chain_length():
    """
    Determines the length of the email chain based on weighted probabilities.
    """
    weights = [0.75, 0.15, 0.08, 0.02]
    chain_length = np.random.choice(range(1, 5), p=weights)
    return chain_length


In [113]:
# Pick email sender and recievers
email_sender = pick_email_sender_details()
email_receivers_list = pick_email_receivers(email_sender)
email_to_str = make_receiver_string(email_receivers_list)

# Build the original email
text_prompt = create_first_email_text_prompt(email_sender, email_receivers_list)
email_gen = generate_email_body_llm_response(text_prompt)
email_body = extract_email_body(email_gen)
header_text = build_email_header(email_sender, sent_date, email_to_str)
email_text = combine_header_body(header_text, email_body)
full_email = ensure_blank_lines_around_categories(email_text)

# Build email chain
chain_len = determine_chain_length()
if chain_len > 1:
    for iteration in range(1,chain_len+1):
        print(iteration)


Ed Burris; Larry Goins; Cameron Howe; Barry Shields; Donna Clark; Yo-Yo Engberk; Malcolm Levitan; Gordon Clark; Debbie Malinowski


In [81]:
def generate_reply_text_prompt(previous_email, reply_sender_dict, emotion, category):
    """
    Generates a reply to the previous email.
    """
    text_prompt = (
        "Here is an email that was generated. Please generate a reply from the character {} in their voice and personality from the tv series Halt and Catch Fire?\n"
        "Right now they are feeling {}.  The topic they are writing about in thier reply is {}. Here is the text of the previous email they are replying to:"
        "\n```\n{}\n```\n"
        "Please DO NOT include Attachments:, Categories:, To:, From:, BCc, or Cc. I will add that afterwards.\n"
        "Please DO NOT include a Subject: \n"
        "Please start the email with 'EMAIL START' \n"
        "Please end the email with 'EMAIL END' after the signature block.  \n"
        "The email should have three parts:\n"
        "The first part is the greeting. You can use 'Dear','Hi','Hello', or whatever you deem is appropriate for the kind of conersation you are writing. \n"
        "The second part is the body of the email.\n"
        "The fourth part is the salutation before the signature block.\n"
        "The saluation should be 'Very truly yours', 'Cheers!', 'Regards', 'Best Regards', 'Warm Regards', or whatever you deem is appropriate for the kind of conersation you are writing. \n" 
        "The signature block should follow the salutation and have the sender's name, which is '{}'\n, the phone number, which is '{}'.\n, the title, which is '{}',\n the department which is,'{}'\n and the company is '{}'.\n"
        "As a reminder, please start the email with 'EMAIL START' \n"
        "As a reminder, please end the email with 'EMAIL END after the signature block.'  \n" 
        "Thank you so much for your help on this! I really appreciate it!").format(
        reply_sender_dict['name'],
        emotion,
        category,
        previous_email,
        reply_sender_dict['name'],
        reply_sender_dict['Phone'],
        reply_sender_dict['Title'],
        reply_sender_dict['Company'],
        reply_sender_dict['Department'],
        )
    
    return text_prompt

In [82]:
def update_email_receivers(email_receivers_list, email_sender, reply_sender):
    # Remove reply_sender from the list if present
    if reply_sender in email_receivers_list:
        email_receivers_list.remove(reply_sender)
    
    # Add email_sender to the list if not already present
    if email_sender not in email_receivers_list:
        email_receivers_list.append(email_sender)

    return email_receivers_list

In [83]:
def build_reply_email_header(reply_character_name, sent, email_to_str):
    header_text = (
        "\n"
        "From:    {}\n"
        "Sent:    {}\n"
        "To:      {}\n"
    ).format(reply_character_name, sent, email_to_str)
    return header_text

In [84]:
# Function to randomly pick a character and associated attributes
def pick_reply_sender_details(email_receivers_list):

    # Randomly pick a character
    characters = load_json('main_cardiff_characters.json')
    character = random.choice(characters)

    # Randomly pick a topic the character writes about
    topic = random.choice(character['Writes About'])
    
    # Randomly pick a writing emotion
    emotions = load_json('writing_emotions.json')
    emotion = random.choice(emotions['emotions'])

    return {
        'Name': character['name'],
        'Email': character['Email'],
        'Title': character['Title'],
        'Company': character['Company'],
        'Department': character['Department'],
        'Phone': character['Phone'],
        'Quote': character['Quote'],
        'Writes About': topic,
        'Personality Trait': character['Personality Trait'],
        'Writing Style': character['Writing Style'],
        'Emotion': emotion
    }

In [85]:
def build_reply_email_header(reply_character_name, sent, email_to_str, email_category):
    header_text = (
        "\n"
        "From:    {}\n"
        "Sent:    {}\n"
        "To:      {}\n\n"
        "Categories:    {}\n\n"
    ).format(reply_character_name, sent, email_to_str, email_category)
    return header_text

In [32]:
sent_date = email_dates[1]


# Determine the chain length of the email
weights = [0.75, 0.15, 0.08, 0.02]
chain_len = np.random.choice(range(1, 5), p=weights)
print(f'     - chain length: {chain_len}')
print('     - iteration 1')

# Pick email sender and recievers
email_sender_dict = pick_email_sender_details()
email_receivers_list = pick_email_receivers(email_sender_dict)
email_to_str = make_receiver_string(email_receivers_list)

# Build the original email
text_prompt = create_first_email_text_prompt(email_sender_dict, email_receivers_list)
email_gen = generate_email_body_llm_response(text_prompt)
email_body = extract_email_body(email_gen)
header_text = build_email_header(email_sender_dict, sent_date, email_to_str)
email_text = combine_header_body(header_text, email_body)
full_email = ensure_blank_lines_around_categories(email_text)
full_email = full_email.rstrip('E ')  # Strip trailing E

characters_list = [
    "Joe MacMillan", "Gordon Clark", "Cameron Howe", "Donna Clark", "John Bosworth", 
    "Malcolm Levitan", "Yo-Yo Engberk", "Barry Shields", "Debbie Malinowski", "Larry Goins", "Ed Burris"
    ]

# Generate the email replies if the chain length is greater than 1
previous_email = full_email
if chain_len > 1:
    for iteration in range(1,chain_len):

        # Determine the character who is going to reply to the email. 
        reply_character_name = random.choice(email_receivers_list)

        # Obtain the character's integer location 
        position_idx = characters_list.index(reply_character_name)
        characters_json = load_json('main_cardiff_characters.json')
        reply_sender_dict = characters_json[position_idx]

        # Randomly pick the character's emotional state
        emotions = load_json('writing_emotions.json')
        emotion = random.choice(emotions['emotions'])
        
        # Update the email_receivers_list to put the original sender in the recievers list and take out the char replying.
        email_sender = email_sender_dict['Name']
        email_receivers_list = update_email_receivers(email_receivers_list, email_sender, reply_character_name)

        # Generate reply email text
        reply_text_prompt = generate_reply_text_prompt(previous_email, reply_sender_dict, emotion)
        reply_email_gen = generate_email_body_llm_response(reply_text_prompt)
        reply_email_body = extract_email_body(reply_email_gen)
        email_receivers_str = make_receiver_string(email_receivers_list)
        reply_header_text = build_reply_email_header(reply_character_name, sent_date, email_receivers_str)
        reply_email_text = combine_header_body(reply_header_text, reply_email_body)
        full_reply_email = ensure_blank_lines_around_categories(reply_email_text)        
        full_reply_email = full_reply_email.rstrip('E ')  # Strip trailing E

        previous_email = ('{}\n{}'.format(full_reply_email,previous_email))
        print(f'     - iteration {iteration+1}')

print(previous_email)

     - chain length: 1
     - iteration 1

From:    Larry Goins
Sent:    Sunday, January 02, 1983 01:49 PM
To:      Barry Shields; Joe MacMillan; Debbie Malinowski
Subject: System Check-Ups

Categories: Wellness Check

Dear Barry, Joe, and Debbie,

I'm reaching out to make sure our systems are running smoothly. As part of our regular wellness checks, I wanted to highlight a few key areas where you can take proactive steps to ensure your machines are humming along.

Firstly, have you considered running the "System Refresh" tool on your computers? This will help tidy up any loose ends and prevent errors from creeping in. If you're experiencing any issues or slow performance, please don't hesitate to reach out – we're here to support you.

Additionally, I recommend taking a few minutes each week to review system logs for any unusual activity. This can help identify potential problems before they become major headaches.

Lastly, if you have any questions or concerns about your systems' hea

In [88]:
def generate_email_chain(sent_date):

    # Determine the chain length of the email
    weights = [0.75, 0.12, 0.08, 0.02, 0.01, 0.01, 0.01]
    chain_len = np.random.choice(range(1, 8), p=weights)
    # print(f'     - chain length: {chain_len}')
    # print('     - iteration 1')

    # Pick email sender and recievers
    email_sender_dict = pick_email_sender_details()
    email_category = email_sender_dict['Writes About']
    email_receivers_list = pick_email_receivers(email_sender_dict)
    email_to_str = make_receiver_string(email_receivers_list)

    # Build the original email
    text_prompt = create_first_email_text_prompt(email_sender_dict, email_receivers_list)
    email_gen = generate_email_body_llm_response(text_prompt)
    email_body = extract_email_body(email_gen)
    header_text = build_email_header(email_sender_dict, sent_date, email_to_str)
    email_text = combine_header_body(header_text, email_body)
    full_email = ensure_blank_lines_around_categories(email_text)
    full_email = full_email.rstrip('E ')  # Strip trailing E

    characters_list = [
        "Joe MacMillan", "Gordon Clark", "Cameron Howe", "Donna Clark", "John Bosworth", 
        "Malcolm Levitan", "Yo-Yo Engberk", "Barry Shields", "Debbie Malinowski", "Larry Goins", "Ed Burris"
        ]

    # Generate the email replies if the chain length is greater than 1
    previous_email = full_email
    if chain_len > 1:
        for iteration in range(1,chain_len):

            # Determine the character who is going to reply to the email. 
            reply_character_name = random.choice(email_receivers_list)

            # Obtain the character's integer location 
            position_idx = characters_list.index(reply_character_name)
            characters_json = load_json('main_cardiff_characters.json')
            reply_sender_dict = characters_json[position_idx]

            # Randomly pick the character's emotional state
            emotions = load_json('writing_emotions.json')
            emotion = random.choice(emotions['emotions'])
        
            # Update the email_receivers_list to put the original sender in the recievers list and take out the char replying.
            email_sender = email_sender_dict['Name']
            email_receivers_list = update_email_receivers(email_receivers_list, email_sender, reply_character_name)

            # Generate reply email text
            reply_text_prompt = generate_reply_text_prompt(previous_email, reply_sender_dict, emotion, email_category)
            reply_email_gen = generate_email_body_llm_response(reply_text_prompt)
            reply_email_body = extract_email_body(reply_email_gen)
            email_receivers_str = make_receiver_string(email_receivers_list)
            reply_header_text = build_reply_email_header(reply_character_name, sent_date, email_receivers_str, email_category)
            reply_email_text = combine_header_body(reply_header_text, reply_email_body)
            full_reply_email = ensure_blank_lines_around_categories(reply_email_text)        
            full_reply_email = full_reply_email.rstrip('E ')  # Strip trailing E

            previous_email = ('{}\n{}'.format(full_reply_email,previous_email))
            #print(f'     - iteration {iteration+1}')

    return previous_email, chain_len, len(previous_email)

In [87]:
import time

# RUN HERE

In [89]:
def save_emails(dates, directory="emails"):
    """
    Saves emails for each date in the list to sequentially numbered files.
    
    Args:
    dates (list of str): A list of date strings.
    directory (str): The directory where files will be saved.
    """
    # Ensure the directory exists
    if not os.path.exists(directory):
        os.makedirs(directory)
        
    times = []
    
    # Iterate over dates and save each email to a separate file
    for i, date in enumerate(dates, start=1):
        
        # Timing the function
        start_time = time.time()

        # Main operation
        filename = f"{directory}/{i:05d}.txt"
        email_content, chain_len, email_len = generate_email_chain(sent_date=date)
        
        with open(filename, 'w', encoding='utf-8') as file:
            file.write(email_content)
        
        # Stopping time
        end_time = time.time()

        # Calculate the elapsed time
        elapsed_time = end_time - start_time
        times.append(elapsed_time)
        
        print(f'Saved file: {filename}   Time: {elapsed_time}   Chain Length: {chain_len}   Email Length:   {email_len}')
        
        if i == 99999:  # Stop after 99999 files
            break
            
    # Calculate the mean
    mean_value = sum(times) / len(times)
    print(f'Mean value: {mean_value}')
    
    return times, mean_value

In [75]:
# Execute Emails
email_dates = generate_email_dates()
print(len(email_dates))
times, mean_value = save_emails(dates=email_dates, directory="emails")

9544
Saved file: emails/00001.txt   Time: 88.41489720344543
Saved file: emails/00002.txt   Time: 37.32603168487549
Saved file: emails/00003.txt   Time: 32.33577108383179
Saved file: emails/00004.txt   Time: 38.357656955718994
Saved file: emails/00005.txt   Time: 40.9142119884491
Saved file: emails/00006.txt   Time: 36.44586777687073
Saved file: emails/00007.txt   Time: 29.155027151107788
Saved file: emails/00008.txt   Time: 31.85168147087097
Saved file: emails/00009.txt   Time: 31.64986538887024
Saved file: emails/00010.txt   Time: 39.06983804702759
Saved file: emails/00011.txt   Time: 87.45002889633179
Saved file: emails/00012.txt   Time: 40.36227369308472
Saved file: emails/00013.txt   Time: 38.74183797836304
Saved file: emails/00014.txt   Time: 29.59276008605957
Saved file: emails/00015.txt   Time: 39.77179312705994
Saved file: emails/00016.txt   Time: 35.02377438545227
Saved file: emails/00017.txt   Time: 38.244882345199585
Saved file: emails/00018.txt   Time: 44.47975945472717
Sav

KeyboardInterrupt: 

In [ ]:
STOP

In [29]:
email_receivers_list

['Ed Burris']

In [30]:
# Determine the character who is going to reply to the email. 
reply_character_name = random.choice(email_receivers_list)
email_receivers_str = make_receiver_string(email_receivers_list)
reply_header_text = build_reply_email_header(reply_character_name, sent_date, email_receivers_str)
print(reply_header_text)


From:    Ed Burris
Sent:    Sunday, January 02, 1983 01:49 PM
To:      Ed Burris



In [118]:
reply_sender = random.choice(email_receivers_list)

character_dict = {character[reply_sender]: character for character in characters}
character_dict


KeyError: 'Donna Clark'

In [123]:

characters_list = [
    "Joe MacMillan", "Gordon Clark", "Cameron Howe", "Donna Clark", "John Bosworth", 
    "Malcolm Levitan", "Yo-Yo Engberk", "Barry Shields", "Debbie Malinowski", "Larry Goins", "Ed Burris", 
    ]

In [100]:
email_1 = generate_initial_email(email_dates[1])
text_prompt = generate_reply_email(previous_email=email_1)
email_2 = generate_email_body_llm_response(text_prompt)
print(email_2)

I've generated a reply from John Bosworth, one of the recipients of the original email. Here it is:

EMAIL START
Categories: Industry News

Hey Malcolm,

Thanks for the update on industry trends and code snippets! Always great to get insights on what's happening in the market.

The article about IBM PCjr's potential impact on Apple II's market share got me thinking. It's definitely something we should keep an eye on, especially since our own project is heavily reliant on Apple II technology. I agree that it's worth considering how this development might shape our strategy moving forward.

As for the code snippet you shared, thanks for passing it along! I'll definitely give it a try and see if it improves our database query efficiency. You're right, optimization is key in our line of work.

Thanks again for keeping us informed and sharing your expertise!

EMAIL END


In [98]:
print(email_2)

Here is a reply to Debbie's email:

EMAIL START
Categories: Appreciation for Your Hard Work

Dear Debbie,

Thank you so much for your kind words and recognition of our team's efforts. We're thrilled to be making progress on Cardiff Electric's projects, and we appreciate your leadership and support throughout the process.

Please know that we're committed to continuing our collaboration and expertise to drive initiatives forward. We're excited about the prospect of pushing boundaries together!

EMAIL END

Note: I kept the response concise and focused on acknowledging Debbie's appreciation and expressing the team's commitment to their work. The signature block is not included as per your request, but it would typically include the recipient's name, title, email address, phone number, and other contact information.


In [82]:
email = generate_initial_email(email_dates[1])
print(email)


From:    John Bosworth
Sent:    Monday, January 03, 1983 12:12 PM
To:      Donna Clark; 
Subject: Closing Deals and Moving Forward

Categories: Thank You

Dear Donna,

Let's be real, we're not just closing deals, we're changing lives. And I'm grateful for your role in making that happen. Your hard work and dedication to Cardiff Electric are paying off, and it shows.

As we head into the home stretch of Q2, I want to make sure you know how much I appreciate everything you've done so far. From pushing the limits of what's possible with our products to being a rock-solid partner for your colleagues, you're an invaluable asset to this team.

Keep up the fantastic work, Donna. You're making a real difference here.

Best regards,
John Bosworth
Vice President of Sales
Cardiff Electric | Sales
+1 512-555-0179



In [94]:
email_1 = """

From:    John Bosworth
Sent:    Monday, January 03, 1983 12:12 PM
To:      Donna Clark; 
Subject: Closing Deals and Moving Forward

Categories: Thank You

Dear Donna,

Let's be real, we're not just closing deals, we're changing lives. And I'm grateful for your role in making that happen. Your hard work and dedication to Cardiff Electric are paying off, and it shows.

As we head into the home stretch of Q2, I want to make sure you know how much I appreciate everything you've done so far. From pushing the limits of what's possible with our products to being a rock-solid partner for your colleagues, you're an invaluable asset to this team.

Keep up the fantastic work, Donna. You're making a real difference here.

Best regards,
John Bosworth
Vice President of Sales
Cardiff Electric | Sales
+1 512-555-0179
"""
email_2 = """

From:    Donna Clark
Sent:    Monday, January 03, 1983 2:45 PM
To:      John Bosworth; 
Subject: Re: Closing Deals and Moving Forward

Dear John,

Wow, thank you so much for your kind words! It truly means a lot to me to 
hear that my efforts are making a difference. I have to say, it's an 
incredible feeling knowing that the work we're doing is having a positive 
impact on people's lives.

I'm thrilled to be a part of this team and grateful for the opportunity to 
contribute to Cardiff Electric's success. Your leadership and guidance 
have been instrumental in my growth and development, and I appreciate your 
support and trust in me.

Thank you again for your recognition and encouragement. It motivates me to 
continue striving for excellence and making a positive impact.

Best regards,
Donna Clark


"""
full_email = email_1+email_2
print(full_email)



From:    John Bosworth
Sent:    Monday, January 03, 1983 12:12 PM
To:      Donna Clark; 
Subject: Closing Deals and Moving Forward

Categories: Thank You

Dear Donna,

Let's be real, we're not just closing deals, we're changing lives. And I'm grateful for your role in making that happen. Your hard work and dedication to Cardiff Electric are paying off, and it shows.

As we head into the home stretch of Q2, I want to make sure you know how much I appreciate everything you've done so far. From pushing the limits of what's possible with our products to being a rock-solid partner for your colleagues, you're an invaluable asset to this team.

Keep up the fantastic work, Donna. You're making a real difference here.

Best regards,
John Bosworth
Vice President of Sales
Cardiff Electric | Sales
+1 512-555-0179


From:    Donna Clark
Sent:    Monday, January 03, 1983 2:45 PM
To:      John Bosworth; 
Subject: Re: Closing Deals and Moving Forward

Dear John,

Wow, thank you so much for your kind

In [84]:
# Avg time to generate an email. 
import time

times = []

for i in range(0,9):
    # Timing the function
    start_time = time.time()
    gen_email = generate_initial_email(sent)
    end_time = time.time()
    
    # Calculate the elapsed time
    elapsed_time = end_time - start_time
    times.append(elapsed_time)

    print(f"The function took {elapsed_time} seconds to run.")
    
# Calculate the mean
mean_value = sum(times) / len(times)

# Print the result
print("The mean is:", mean_value)    

NameError: name 'sent' is not defined

In [85]:
def save_emails(dates, directory="emails"):
    """
    Saves emails for each date in the list to sequentially numbered files.
    
    Args:
    dates (list of str): A list of date strings.
    directory (str): The directory where files will be saved.
    """
    # Ensure the directory exists
    if not os.path.exists(directory):
        os.makedirs(directory)
        
    times = []
    
    # Iterate over dates and save each email to a separate file
    for i, date in enumerate(dates, start=1):
        
        # Timing the function
        start_time = time.time()

        # Main operation
        filename = f"{directory}/{i:05d}.txt"
        email_content, email_sender = generate_initial_email(sent=date)
        
        with open(filename, 'w', encoding='utf-8') as file:
            file.write(email_content)
        
        # Stopping time
        end_time = time.time()

        # Calculate the elapsed time
        elapsed_time = end_time - start_time
        times.append(elapsed_time)
        
        print(f'Saved file: {filename}   Time: {elapsed_time}')
        
        if i == 99999:  # Stop after 99999 files
            break
            
    # Calculate the mean
    mean_value = sum(times) / len(times)
    print(f'Mean value: {mean_value}')
    
    return times, mean_value

In [89]:
emails_today = np.random.randint(0, 3)
emails_today

1

In [90]:
data_path = "/home/aaronnhorvitz/dev/work/irs/TopicMiner/data/fake_data_generator/new_emails"
dates = generate_email_dates()
len(dates)

5820

In [442]:
times, mean_value = save_emails(dates, directory="emails")

Saved file: emails/00001.txt   Time: 31.631754875183105
Saved file: emails/00002.txt   Time: 34.90022611618042
Saved file: emails/00003.txt   Time: 34.83442687988281
Saved file: emails/00004.txt   Time: 25.698625087738037
Saved file: emails/00005.txt   Time: 36.794296741485596
Saved file: emails/00006.txt   Time: 32.16952729225159
Saved file: emails/00007.txt   Time: 30.808124542236328
Saved file: emails/00008.txt   Time: 33.49370098114014
Saved file: emails/00009.txt   Time: 33.14686155319214
Saved file: emails/00010.txt   Time: 34.04681658744812
Saved file: emails/00011.txt   Time: 28.736342191696167
Saved file: emails/00012.txt   Time: 29.474615573883057
Saved file: emails/00013.txt   Time: 30.401808500289917
Saved file: emails/00014.txt   Time: 33.62435698509216
Saved file: emails/00015.txt   Time: 25.909223556518555
Saved file: emails/00016.txt   Time: 33.523128271102905
Saved file: emails/00017.txt   Time: 41.40921068191528
Saved file: emails/00018.txt   Time: 33.037580490112305


In [ ]:
STOP

In [420]:
# Avg time to generate an email. 
import time

times = []

for i in range(0,9):
    # Timing the function
    start_time = time.time()
    gen_email = generate_initial_email(sent)
    end_time = time.time()
    
    # Calculate the elapsed time
    elapsed_time = end_time - start_time
    times.append(elapsed_time)

    print(f"The function took {elapsed_time} seconds to run.")
    
# Calculate the mean
mean_value = sum(times) / len(times)

# Print the result
print("The mean is:", mean_value)    

The function took 28.794631958007812 seconds to run.
The function took 24.696451902389526 seconds to run.
The function took 28.61640191078186 seconds to run.
The function took 33.9649384021759 seconds to run.
The function took 35.59217023849487 seconds to run.
The function took 31.303925037384033 seconds to run.
The function took 36.33676099777222 seconds to run.
The function took 32.48104786872864 seconds to run.
The function took 31.33132004737854 seconds to run.
The mean is: [28.794631958007812, 24.696451902389526, 28.61640191078186, 33.9649384021759, 35.59217023849487, 31.303925037384033, 36.33676099777222, 32.48104786872864, 31.33132004737854]


In [426]:
sum(times)/len(times)

31.45751648479038

In [422]:
print(gen_email)


From:    Gordon Clark
Sent:    Saturday, December 31, 1983 02:02 PM
To:      John Bosworth; 
Subject: Vacation Notice

Categories: Vacation Notice

Dear John,

As I sit here amidst the hum of our servers and the quiet desperation of our mainframe, my mind wanders to more... leisurely pursuits. I'm writing to inform you that I'll be taking a brief hiatus from June 15th to July 1st. I've made arrangements for coverage during this time, but please don't hesitate to reach out if anything critical arises.

I must admit, the thought of trading my daily dose of hexadecimal code for some warm sun and distant horizons is tantalizing. Perhaps, in the grand tapestry of things, a break from the usual routine will allow me to approach our engineering challenges with renewed vigor when I return.

Gordon Clark
Chief Engineer
Cardiff Electric
Engineering Department
+1 512-555-0198

E


In [439]:
print(gen_email.rstrip('E'))


From:    Gordon Clark
Sent:    Saturday, December 31, 1983 02:02 PM
To:      John Bosworth; 
Subject: Vacation Notice

Categories: Vacation Notice

Dear John,

As I sit here amidst the hum of our servers and the quiet desperation of our mainframe, my mind wanders to more... leisurely pursuits. I'm writing to inform you that I'll be taking a brief hiatus from June 15th to July 1st. I've made arrangements for coverage during this time, but please don't hesitate to reach out if anything critical arises.

I must admit, the thought of trading my daily dose of hexadecimal code for some warm sun and distant horizons is tantalizing. Perhaps, in the grand tapestry of things, a break from the usual routine will allow me to approach our engineering challenges with renewed vigor when I return.

Gordon Clark
Chief Engineer
Cardiff Electric
Engineering Department
+1 512-555-0198




In [412]:
gen_email = generate_initial_email(sent)
print(gen_email)


From:    Yo-Yo Engberk
Sent:    Saturday, December 31, 1983 02:02 PM
To:      Larry Goins; Malcolm Levitan; John Bosworth; 
Subject: The Quest for Better Supplier Relations Begins!

Categories: Supplier Relations


Dear Larry, Malcolm, and John,

I hope this email finds you well-oiled machines, churning out code like there's no tomorrow. Or should I say, like there's no 1984?

On a more serious note (just for a sec, I promise), I wanted to touch base about our supplier relations. As we all know, Cardiff Electric is only as strong as its weakest link – and that link might just be our supply chain. So, let's get this party started! Who's up for some quality time with our vendors?

Cheers,
Yo-Yo Engberk
Programmer, Software Development
Cardiff Electric
+1 512-555-0142

E


In [413]:
save_text_to_file(text=gen_email, filename="/home/aaronnhorvitz/dev/work/irs/TopicMiner/data/fake_data_generator/new_emails/example.txt")

In [391]:
pwd()

'/home/aaronnhorvitz/dev/work/irs/TopicMiner/data/fake_data_generator'

In [336]:
header_text

'\nFrom:    Barry Shields\nSent:    Saturday, December 31, 1983 02:02 PM\nTo:      Cameron Howe, John Bosworth, Donna Clark, \n'

In [337]:
email_body

"Subject:    Personal Milestone\nCategories:    Personal Milestone\nDear Cameron, John, Donna,\n\nAs I reflect on the past quarter, I am pleased to report that I have successfully navigated a critical juncture in my personal and professional development. Specifically, I have reached a milestone of 500 hours spent studying the intricacies of relevant case law, thereby enhancing my expertise in areas pertinent to Cardiff Electric's interests.\n\nI must confess that this achievement has brought me a sense of accomplishment and pride, particularly given the rigorous standards to which I hold myself accountable. It is a testament to the value I place on continuous learning and self-improvement, both as an individual and as In-House Counsel for our esteemed organization.\n\nPlease join me in celebrating this personal milestone, and know that I remain committed to serving Cardiff Electric's legal needs with unwavering dedication and professionalism.\n\nSincerely,\nBarry Shields\n+1 512-555-01

In [342]:
email_text = "{}{}".format(header_text, email_body) 
print(email_text)


From:    Barry Shields
Sent:    Saturday, December 31, 1983 02:02 PM
To:      Cameron Howe, John Bosworth, Donna Clark, 
Subject:    Personal Milestone
Categories:    Personal Milestone
Dear Cameron, John, Donna,

As I reflect on the past quarter, I am pleased to report that I have successfully navigated a critical juncture in my personal and professional development. Specifically, I have reached a milestone of 500 hours spent studying the intricacies of relevant case law, thereby enhancing my expertise in areas pertinent to Cardiff Electric's interests.

I must confess that this achievement has brought me a sense of accomplishment and pride, particularly given the rigorous standards to which I hold myself accountable. It is a testament to the value I place on continuous learning and self-improvement, both as an individual and as In-House Counsel for our esteemed organization.

Please join me in celebrating this personal milestone, and know that I remain committed to serving Cardiff E

In [343]:
full_email = ensure_blank_lines_around_categories(email_text)
print(full_email)


From:    Barry Shields
Sent:    Saturday, December 31, 1983 02:02 PM
To:      Cameron Howe, John Bosworth, Donna Clark, 
Subject:    Personal Milestone

Categories:    Personal Milestone

Dear Cameron, John, Donna,

As I reflect on the past quarter, I am pleased to report that I have successfully navigated a critical juncture in my personal and professional development. Specifically, I have reached a milestone of 500 hours spent studying the intricacies of relevant case law, thereby enhancing my expertise in areas pertinent to Cardiff Electric's interests.

I must confess that this achievement has brought me a sense of accomplishment and pride, particularly given the rigorous standards to which I hold myself accountable. It is a testament to the value I place on continuous learning and self-improvement, both as an individual and as In-House Counsel for our esteemed organization.

Please join me in celebrating this personal milestone, and know that I remain committed to serving Cardiff

In [321]:
email_sender = pick_email_sender_details()
email_receivers_list = pick_email_receivers(email_sender)
email_to_str = make_receiver_string(email_receivers_list)
text_prompt = create_first_email_text_prompt(email_sender, email_receivers_list)
email_generation_text = generate_email_body_llm_response(text_prompt)
email_body = extract_email_body(email_generation_text)
print(email_body)

Subject:    Personal Milestone
Categories:    Personal Milestone
Dear Cameron, John, Donna,

As I reflect on the past quarter, I am pleased to report that I have successfully navigated a critical juncture in my personal and professional development. Specifically, I have reached a milestone of 500 hours spent studying the intricacies of relevant case law, thereby enhancing my expertise in areas pertinent to Cardiff Electric's interests.

I must confess that this achievement has brought me a sense of accomplishment and pride, particularly given the rigorous standards to which I hold myself accountable. It is a testament to the value I place on continuous learning and self-improvement, both as an individual and as In-House Counsel for our esteemed organization.

Please join me in celebrating this personal milestone, and know that I remain committed to serving Cardiff Electric's legal needs with unwavering dedication and professionalism.

Sincerely,
Barry Shields
+1 512-555-0166
In-House C

In [308]:
for i in range(0,50):
    print(f"\nEmail {i}: ---------|\n")
    email_sender = pick_email_sender_details()
    email_receivers_list = pick_email_receivers(email_sender)
    email_to_str = make_receiver_string(email_receivers_list)
    text_prompt = create_first_email_text_prompt(email_sender, email_receivers_list)
    email_generation_text = generate_email_body_llm_response(text_prompt)
    email_body = extract_email_body(email_generation_text)
    print(email_body)
    


Email 0: ---------|

Subject:    Product Recall - Critical Issues in 286

Attachments:   

Categories:    product_recall

Dear Malcolm,

As I sit here, staring at the code, I'm forced to confront the abyss that lies before us. Our 286 prototype has taken a turn for the worse, and I fear we may have to recall the entire batch. The errors are too numerous, the flaws too profound. It's as if Cardiff Electric has been sleepwalking through innovation, and now we're paying the price.

The issues are multifaceted: faulty logic gates, misaligned clock signals, and a general lack of coherence in the design. I've tried to reason with my colleagues, but they seem oblivious to the impending doom that awaits us. It's as if they're trapped in some sort of bureaucratic haze, unable to see the forest for the trees.

I know I should be optimistic, but I just can't muster the enthusiasm. We've poured our hearts and souls into this project, only to have it turn on us like a rabid animal. I'm at my wit's